# groupe 
## JIE PENGYU && KUSUMA KOTTETI

| Field                                | Content (Draft)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          |
| :----------------------------------- | :----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Task / Story**                     | Build a **web-based dashcam capture client** (browser camera) that generates **perceptual hashes on-device** and streams **hashes+metadata** to Supabase, plus an **authenticated (Angular) validator portal** that verifies an uploaded accident video by re-hashing sampled frames and comparing via **Hamming distance** against Supabase data (tolerant to re-encoding).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| **User Value**                       | Enables an insurance-like workflow to **detect tampering** (trim/insert/re-encode/fps changes) by comparing a submitted video to an independently synced “hash timeline” captured earlier; showcases real-time sync + offline buffering + short retention in a prototype.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                |
| **Acceptance Criteria (Functional)** | **Capture (Web Client)**<br>- Uses browser camera on laptop/phone to record a session.<br>- On-device: samples frames using **timestamp-based sampling** with a **variable sampling interval** (configurable).<br>- For each sampled frame: compute **pHash-like perceptual hash** and store `{sessionId, sampleTimestamp, hash, algoVersion, intervalUsed, deviceClockStart, …}` locally + attempt upload to Supabase immediately (“on-the-fly”).<br>- **Offline buffering**: if upload fails (no network), persist unsent records locally; when connection resumes, resume upload **from the earliest unsent timestamp** until caught up.<br><br>**Backend / Storage (Supabase)**<br>- Supabase stores **hashes + metadata only** (no frames/video).<br>- Data is segregated per user/session; server enforces auth-based access (no public writes/reads).<br>- Retention: Supabase scheduled job deletes old session/hash data after a short window (prototype).<br><br>**Validator Portal (Angular + C# API)**<br>- Requires authentication (insurance user / validator role).<br>- Claim submission: upload AVI video (25/30fps possible, may be re-encoded).<br>- Server-side: determine recording start time based on **device clock at recording start** (provided in metadata); derive time window; re-sample frames by timestamps; compute same perceptual hash algorithm; compare with Supabase hashes using **Hamming distance threshold**.<br>- Outputs verdict: **Verified / Suspicious / Inconclusive** with metrics: match ratio, missing segments, max/avg distance, detected trims/inserts (at least coarse indicators).<br><br>**MVP sequencing**<br>- Version 1: happy path end-to-end works (capture→Supabase→upload→verdict).<br>- Version 2: offline buffering + re-sync robustness demonstrated. |
| **Technical Constraints**            | - Frontend portal: **Angular**.<br>- Backend: **C#** (assume ASP.NET Core) for validator API / processing.<br>- Storage/Auth/cron: **Supabase** (Postgres + Auth + scheduled job).<br>- Hashing: **perceptual hashing (pHash suggested)**, comparison via **Hamming distance** (tolerant to compression/re-encoding).<br>- Sampling: **timestamp-based**, interval is **variable/configurable**.<br>- Time-box: **17 hours / 7 days**, 2-person team, GitHub workflow + commit summaries.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                |
| **Invariants**                       | - Hashing is done **on-device during recording** (not server-generated for source-of-truth).<br>- Supabase stores **only hashes+metadata** (no raw frames/video).<br>- Matching uses **Hamming distance** (not exact equality) to tolerate re-encoding.<br>- Time reference anchor: **device clock at recording start**.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| **Definition of Done (DoD)**         | - **Demo script** exists and is repeatable: (1) start capture session, (2) observe hashes appearing in Supabase in near real-time, (3) upload AVI, (4) portal shows verdict + match metrics.<br>- **Happy-path MVP** completed first (end-to-end).<br>- **Offline buffering demo**: disconnect network mid-session, continue sampling locally, reconnect, verify backlog uploads and validator can still verify a video spanning the offline period.<br>- **Retention**: scheduled deletion runs (or can be triggered) and validator gracefully reports “data expired/not found”.<br>- **Auth**: unauthenticated users cannot read/write hashes; validator portal requires login.<br>- Basic QA checklist (manual) covers: sampling interval changes, re-encode tolerance, fps difference, trim/insert simulation, network loss recovery.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                |

## Summary of Progress
    today until now we did little search found what we defined our tasks, story, conference, what we need how to preoceed this code, we wrote the Goal on what to do, We worked on Constraints we needed, we both divided our work for the frontend(Kusuma) and for the backend(jay) will be working and we also decided to track our progress every single day what we have worked on and, we checked if our plan is suitable or not otherwise we need to adjust our planning thats why we are following this process and next me moved to prototype experiment firstly we generate ground truth hash timeline, create tamper variants automatically, and then we re-sample and re-hash Variants, we compared with hamming distance  and timestamp tolerance so we verified re-encoded and fps changed variant still yields a high match ratio with a stable distance threshold and small timestamp tolerance window，

## results of spike experiment
    dHash 64-bit (9×8 grayscale) is stable under re-encode and 30→25 fps at 500ms sampling and ±200ms tolerance.
    Distances are tight: p90=4, p95=4 for both variants, and match ratios are 96.8% (re-encode) and 100% (fps-25) at threshold 4.
    A slightly looser threshold (≤5) pushes re-encode to 98.7%, and ≤8 reaches 100%.
    So: the core verification loop is feasible without a future rewrite.

```powershell
PS D:\Github\eSIGELEC_wOrk\cloud_computing\dashcam-cloudservice-web> npx tsx .\spike_hash_pipeline.ts
Need to install the following packages:
tsx@4.21.0
Ok to proceed? (y) y

Input: d:\Github\eSIGELEC_wOrk\cloud_computing\dashcam-cloudservice-web\test_blender.mp4
Duration: 78413 ms
Interval: 500 ms, Tolerance: 200 ms
Hash: dHash (9x8 grayscale, 64-bit)

Generating ground truth timeline...
Frames hashed: 157

Creating variants in C:\Users\33573\AppData\Local\Temp\hash-spike-BDfkhg

Sampling variant: re-encode

Sampling variant: fps-25

Distance stats (min distance per reference frame):
re-encode: p50=1, p90=4, p95=4
fps-25: p50=1, p90=4, p95=4

Threshold sweep: match ratio (%)
variant <=5     <=8     <=10    <=12    <=15    <=18    <=22
re-encode       98.7    100.0   100.0   100.0   100.0   100.0   100.0
fps-25  100.0   100.0   100.0   100.0   100.0   100.0   100.0

Suggested threshold range (p90-p95) based on min distances:
re-encode: 4 - 4
fps-25: 4 - 4

Summary (using per-variant p90 as threshold):
variant threshold       match%  avgDist maxDist missingSpans
re-encode       4       96.8    1.86    4       51500-53000ms, 56000-56000ms
fps-25  4       100.0   1.64    4       none

Pass criteria: match ratio >= 80% at small tolerance/threshold.
```